# 第 0 章：PyTorch / einops 入门（热身）

本章是 CS336 作业 1 的**前置热身**，不属于原作业内容，目的是把后面会反复用到的"工具动作"提前练熟，
让你在正式作业里可以把注意力放在模型本身，而不是和张量形状搏斗。

**本章覆盖：**

1. 张量基础：创建、dtype / device、形状变换、广播
2. 索引：基础切片、高级索引、布尔掩码、`gather`、`masked_fill`、`triu/tril`
3. 矩阵运算：`@` / `matmul` 的批量语义、`einsum`
4. einops：`rearrange` / `reduce` / `repeat` / `einsum`
5. `nn.Module`：继承、`nn.Parameter`、`register_buffer`、子模块、`forward` 与 `__call__`、`state_dict`、初始化
6. autograd 与最小训练循环
7. jaxtyping 类型标注与调试习惯

**使用方式：** 按顺序运行本 notebook → 做 `exercises/` 下的练习 → 用 `tests/` 自测：

```bash
uv run pytest warmup/tests -v
```

> 本 notebook 中的例子刻意**不**直接对应作业中的组件（Linear、Embedding、RMSNorm、Attention 等），
> 而是教会你写出它们所需的每一个"零件"。

In [1]:
import torch
from torch import nn
import einops
from einops import rearrange, reduce, repeat

torch.manual_seed(0)
print(torch.__version__, einops.__version__)

2.11.0+cu130 0.8.2


## 1. 张量基础

### 1.1 创建张量与 dtype / device

常用创建函数：`torch.tensor`、`torch.zeros/ones/empty/full`、`torch.randn/rand/randint`、`torch.arange`。

几乎所有创建函数都接受 `device=` 和 `dtype=`。**作业中的模块构造函数都要求接受 `device` 和 `dtype` 并向下传递**，
所以养成"创建参数时把它们传进去"的习惯。

In [2]:
a = torch.zeros(2, 3)                       # 默认 float32, cpu
b = torch.ones(2, 3, dtype=torch.float64)
c = torch.arange(6)                         # int64
d = torch.randn(2, 3, device="cpu")

for name, t in dict(a=a, b=b, c=c, d=d).items():
    print(name, t.shape, t.dtype, t.device)

# 类型/设备转换
print(c.float().dtype, c.to(torch.float16).dtype)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("will use device:", device)

a torch.Size([2, 3]) torch.float32 cpu
b torch.Size([2, 3]) torch.float64 cpu
c torch.Size([6]) torch.int64 cpu
d torch.Size([2, 3]) torch.float32 cpu
torch.float32 torch.float16
will use device: cpu


**小陷阱：** 不同 dtype / device 的张量不能直接一起运算（会报错），
`x.new_zeros(...)`、`torch.zeros_like(x)` 可以创建与 `x` 同 dtype、同 device 的新张量。

In [3]:
x = torch.randn(3, dtype=torch.float64)
print(torch.zeros_like(x).dtype, x.new_ones(2, 2).dtype)

torch.float64 torch.float64


### 1.2 形状变换：view / reshape / transpose / permute / unsqueeze / squeeze

| 操作 | 作用 | 备注 |
|---|---|---|
| `x.view(*shape)` | 改变形状，不复制数据 | 要求内存连续 |
| `x.reshape(*shape)` | 同上，不连续时自动复制 | 更省心 |
| `x.transpose(i, j)` / `x.T` / `x.mT` | 交换两个维度 | 结果通常不连续；`.mT` 交换最后两维 |
| `x.permute(...)` | 任意重排维度 | |
| `x.unsqueeze(i)` / `x[:, None]` | 插入长度 1 的维度 | 广播必备 |
| `x.squeeze(i)` | 去掉长度 1 的维度 | |
| `x.flatten(start, end)` | 合并一段连续维度 | |

In [4]:
x = torch.arange(24).reshape(2, 3, 4)
print(x.shape, x.transpose(1, 2).shape, x.permute(2, 0, 1).shape, x.mT.shape)
print(x.unsqueeze(0).shape, x[:, None].shape, x.flatten(0, 1).shape)

y = x.transpose(1, 2)
print("contiguous?", y.is_contiguous())
try:
    y.view(2, 12)
except RuntimeError as e:
    print("view 失败:", str(e)[:60], "...")
print("reshape OK:", y.reshape(2, 12).shape)

torch.Size([2, 3, 4]) torch.Size([2, 4, 3]) torch.Size([4, 2, 3]) torch.Size([2, 4, 3])
torch.Size([1, 2, 3, 4]) torch.Size([2, 1, 3, 4]) torch.Size([6, 4])
contiguous? False
view 失败: view size is not compatible with input tensor's size and str ...
reshape OK: torch.Size([2, 12])


### 1.3 广播（broadcasting）

规则：**从最后一维开始右对齐**比较，每一维要么相等、要么其中一个是 1、要么其中一个不存在。
长度为 1 的维度会被"虚拟复制"到对方的长度（不真的占内存）。

广播是写出"没有 for 循环"代码的关键。例如：要对一批向量 `(N, d)` 的每个特征做不同缩放 `(d,)`，直接相乘即可。

In [5]:
X = torch.randn(4, 3)        # (N, d)
w = torch.tensor([1., 10., 100.])  # (d,)
print((X * w).shape)          # (4, 3)：w 被广播到每一行

# 用 None 插入新维度来构造"两两组合"
u = torch.arange(3.)          # (3,)
v = torch.arange(4.)          # (4,)
table = u[:, None] + v[None, :]  # (3,1) + (1,4) -> (3,4)
print(table)

torch.Size([4, 3])
tensor([[0., 1., 2., 3.],
        [1., 2., 3., 4.],
        [2., 3., 4., 5.]])


**练习对应：** `exercises/tensor_ops.py` 中的 `batched_outer`、`pairwise_sq_dist`。

## 2. 索引

### 2.1 基础切片与 `...`

`...`（Ellipsis）表示"前面所有维度"，写**支持任意前导 batch 维**的代码时非常有用（作业中很多函数输入形状是 `(..., d)`）。

In [6]:
x = torch.randn(2, 3, 4, 5)
print(x[..., 0].shape)        # 取最后一维第 0 个 -> (2,3,4)
print(x[..., :2].shape)       # (2,3,4,2)
print(x[:, -1].shape)         # (2,4,5)

torch.Size([2, 3, 4])
torch.Size([2, 3, 4, 2])
torch.Size([2, 4, 5])


### 2.2 高级索引（用整数张量做索引）

- `table[idx]`：`idx` 可以是**任意形状**的整数张量，结果形状 = `idx.shape + table.shape[1:]`。
- `x[rows, cols]`：两个形状相同的整数张量配对取值。

In [7]:
table = torch.arange(12.).reshape(4, 3)   # 4 行，每行 3 个数
idx = torch.tensor([[0, 3], [2, 2]])     # 形状 (2, 2)
print(table[idx].shape)                  # (2, 2, 3)

scores = torch.arange(12.).reshape(3, 4)
rows = torch.tensor([0, 1, 2])
cols = torch.tensor([3, 0, 1])
print(scores[rows, cols])                # 取 (0,3),(1,0),(2,1)

torch.Size([2, 2, 3])
tensor([3., 4., 9.])


### 2.3 `torch.gather`

`out = x.gather(dim, index)`：沿 `dim` 维按 `index` 取值，`index` 和 `out` 形状相同，且 `index` 的维数要与 `x` 相同。
与高级索引相比它更"显式"，在多维情形下很好用。

In [8]:
x = torch.tensor([[10., 11., 12.],
                  [20., 21., 22.]])
index = torch.tensor([[2], [0]])         # 每行取一个
print(x.gather(dim=1, index=index))      # [[12.], [20.]]
print(x.gather(1, index).squeeze(-1))    # [12., 20.]

tensor([[12.],
        [20.]])
tensor([12., 20.])


### 2.4 布尔掩码、`masked_fill`、`torch.where`、`triu/tril`

- `torch.triu(m, diagonal=k)` / `torch.tril(m, diagonal=k)`：保留上/下三角。
- `x.masked_fill(mask, value)`：mask 为 True 的位置填 value（返回新张量；`masked_fill_` 才是原地版本）。
- `torch.where(cond, a, b)`：逐元素三元选择。

In [9]:
n = 4
m = torch.ones(n, n, dtype=torch.bool)
print(torch.tril(m).int())               # 下三角（含对角线）
print(torch.triu(m, diagonal=1).int())   # 严格上三角

x = torch.arange(16.).reshape(4, 4)
lower = torch.tril(m)
print(x.masked_fill(~lower, 0.0))        # 只保留下三角
print(torch.where(x > 7, x, -x))

tensor([[1, 0, 0, 0],
        [1, 1, 0, 0],
        [1, 1, 1, 0],
        [1, 1, 1, 1]], dtype=torch.int32)
tensor([[0, 1, 1, 1],
        [0, 0, 1, 1],
        [0, 0, 0, 1],
        [0, 0, 0, 0]], dtype=torch.int32)
tensor([[ 0.,  0.,  0.,  0.],
        [ 4.,  5.,  0.,  0.],
        [ 8.,  9., 10.,  0.],
        [12., 13., 14., 15.]])
tensor([[-0., -1., -2., -3.],
        [-4., -5., -6., -7.],
        [ 8.,  9., 10., 11.],
        [12., 13., 14., 15.]])


**练习对应：** `masked_mean`、`fill_upper_triangle`、`pick_per_row`、`lookup_rows`。

## 3. 矩阵运算

### 3.1 `@` / `torch.matmul` 的批量语义

`A @ B`：最后两维做矩阵乘法，**前面的维度按广播规则处理**。

- `(b, n, k) @ (b, k, m) -> (b, n, m)`
- `(b, n, k) @ (k, m) -> (b, n, m)`（右边被广播到每个 batch）
- 向量 `(k,)` 会被临时当成行/列向量处理

> 线性层的常见写法是 `x @ W.T`，其中 `W` 形状为 `(d_out, d_in)`。想一想这里为什么需要转置，
> 以及当 `x` 的形状是 `(..., d_in)` 时为什么它仍然能工作。

In [10]:
A = torch.randn(5, 2, 3)
B = torch.randn(5, 3, 4)
C = torch.randn(3, 4)
print((A @ B).shape, (A @ C).shape)

W = torch.randn(7, 3)                 # (d_out, d_in)
x = torch.randn(2, 5, 3)              # (..., d_in)
print((x @ W.T).shape)                # (2, 5, 7)

torch.Size([5, 2, 4]) torch.Size([5, 2, 4])
torch.Size([2, 5, 7])


### 3.2 `torch.einsum`

用下标字符串描述运算：**出现在输入但不在输出的下标会被求和**。

| 运算 | einsum |
|---|---|
| 矩阵乘 | `"ik,kj->ij"` |
| 批量矩阵乘 | `"bik,bkj->bij"` |
| 点积 | `"i,i->"` |
| 外积 | `"i,j->ij"` |
| 转置 | `"ij->ji"` |
| 按行求和 | `"ij->i"` |
| 任意前导维 | `"...k,jk->...j"` |

In [11]:
P = torch.randn(3, 4)
Q = torch.randn(4, 5)
torch.testing.assert_close(torch.einsum("ik,kj->ij", P, Q), P @ Q)

u, v = torch.randn(3), torch.randn(3)
print(torch.einsum("i,i->", u, v), u @ v)

x = torch.randn(2, 6, 3)
W = torch.randn(7, 3)
torch.testing.assert_close(torch.einsum("...k,jk->...j", x, W), x @ W.T)
print("einsum OK")

tensor(-3.8317) tensor(-3.8317)
einsum OK


**练习对应：** `batched_matvec`。

## 4. einops：可读的维度操作

einops 的核心思想：**用带名字的维度描述形状变换**，让代码自带文档，并且形状不匹配时会立刻报错。
作业 handout 也推荐使用 einops / einx。

### 4.1 `rearrange`：重排 / 合并 / 拆分维度

- 空格分隔维度名，括号 `( )` 表示合并（或拆分）。
- 拆分时需要通过关键字参数告诉它其中一个子维度的大小。

In [12]:
x = torch.randn(2, 3, 4)                          # (batch, seq, dim)
print(rearrange(x, "b s d -> b d s").shape)       # 转置
print(rearrange(x, "b s d -> (b s) d").shape)     # 合并 -> (6, 4)
print(rearrange(x, "b s (g k) -> b s g k", g=2).shape)   # 拆分 dim=4 -> (2, 2)

# 例子：把一组 RGB 图片 (B, H, W, C) 变成 (B, C, H, W)
imgs = torch.randn(8, 32, 32, 3)
print(rearrange(imgs, "b h w c -> b c h w").shape)

# 例子：把图片沿高度切成 4 条横带，并把每条横带变成一个 batch 元素
imgs = torch.randn(8, 3, 32, 32)
strips = rearrange(imgs, "b c (n h) w -> (b n) c h w", n=4)
print(strips.shape)                              # (32, 3, 8, 32)

torch.Size([2, 4, 3])
torch.Size([6, 4])
torch.Size([2, 3, 2, 2])
torch.Size([8, 3, 32, 32])
torch.Size([32, 3, 8, 32])


**理解括号内顺序：** `(g k)` 表示"g 是外层、k 是内层"，即原来下标 `i = g_idx * k + k_idx`。
`(k g)` 则相反。顺序写错不会报错，但数值会错——这是最常见的 bug，记得用小例子验证！

In [13]:
t = torch.arange(6)
print(rearrange(t, "(g k) -> g k", g=2))   # [[0,1,2],[3,4,5]]
print(rearrange(t, "(k g) -> g k", g=2))   # [[0,2,4],[1,3,5]]

tensor([[0, 1, 2],
        [3, 4, 5]])
tensor([[0, 2, 4],
        [1, 3, 5]])


### 4.2 `reduce` 与 `repeat`

In [14]:
x = torch.randn(2, 3, 4)
print(reduce(x, "b s d -> b d", "mean").shape)    # 沿 s 求均值
print(reduce(x, "b s d -> b", "max").shape)

v = torch.randn(4)
print(repeat(v, "d -> b d", b=3).shape)           # 复制成 (3, 4)

# 图片 2x2 平均池化
img = torch.randn(1, 3, 8, 8)
print(reduce(img, "b c (h 2) (w 2) -> b c h w", "mean").shape)

torch.Size([2, 4])
torch.Size([2])
torch.Size([3, 4])
torch.Size([1, 3, 4, 4])


### 4.3 `einops.einsum`：带名字的 einsum

和 `torch.einsum` 一样，但维度名可以是多字母单词，并且张量放在前面、模式放在最后。

In [15]:
from einops import einsum

x = torch.randn(2, 5, 3)
W = torch.randn(7, 3)
out = einsum(x, W, "batch seq d_in, d_out d_in -> batch seq d_out")
torch.testing.assert_close(out, x @ W.T)

# ... 同样可以用
out2 = einsum(x, W, "... d_in, d_out d_in -> ... d_out")
torch.testing.assert_close(out2, out)
print(out.shape)

torch.Size([2, 5, 7])


**练习对应：** `patchify`、`split_groups`、`merge_groups`。

## 5. `nn.Module`：自定义模块

作业中的每个组件都是一个 `nn.Module` 子类。标准模板：

1. 在 `__init__` 里**第一行**调用 `super().__init__()`（否则后面注册参数会报错）
2. 用 `nn.Parameter(tensor)` 包装可学习参数，并**赋值为 `self` 的属性** —— 这样它才会被自动注册
3. 子模块（另一个 `nn.Module`）同样直接赋值为属性
4. 在 `forward` 中写计算逻辑
5. 使用时调用 `module(x)`，**不要直接调用** `module.forward(x)`（`__call__` 会处理 hooks 等）

In [16]:
class Scaler(nn.Module):
    """y = x * alpha，alpha 是一个可学习的标量"""

    def __init__(self, init: float = 1.0, device=None, dtype=None):
        super().__init__()
        self.alpha = nn.Parameter(torch.tensor(init, device=device, dtype=dtype))
        self.not_a_param = torch.tensor(3.0)        # 普通张量：不会被注册！

    def forward(self, x):
        return x * self.alpha


s = Scaler(2.0)
print(s(torch.ones(3)))
print([name for name, _ in s.named_parameters()])  # 只有 alpha
print(s)

tensor([2., 2., 2.], grad_fn=<MulBackward0>)
['alpha']
Scaler()


### 5.1 参数初始化

推荐模式：先用 `torch.empty(...)` 分配内存，再用 `nn.init` 中的函数**原地**初始化。
作业要求使用截断正态分布 `nn.init.trunc_normal_`，具体的 std 与截断范围请看 handout。

In [17]:
w = torch.empty(4, 3)
nn.init.trunc_normal_(w, mean=0.0, std=0.02, a=-0.04, b=0.04)
print(w.abs().max() <= 0.04)

p = nn.Parameter(torch.empty(5))
nn.init.ones_(p)                    # 对 Parameter 同样可以直接原地初始化
print(p)

tensor(True)
Parameter containing:
tensor([1., 1., 1., 1., 1.], requires_grad=True)


### 5.2 子模块与 `nn.ModuleList`

- 子模块赋值给属性就会被注册，`parameters()` 会递归收集。
- **多个子模块要用 `nn.ModuleList`（或 `nn.ModuleDict`）**存放；放在普通 `list` 里不会被注册，
  它们的参数既不会被优化器看到，也不会出现在 `state_dict` 里。

In [18]:
class Bad(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = [nn.Linear(2, 2) for _ in range(3)]   # ❌ 普通 list

class Good(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.ModuleList([nn.Linear(2, 2) for _ in range(3)])  # ✅

print("Bad  params:", sum(p.numel() for p in Bad().parameters()))
print("Good params:", sum(p.numel() for p in Good().parameters()))
print(list(Good().state_dict().keys())[:4])

Bad  params: 0
Good params: 18
['layers.0.weight', 'layers.0.bias', 'layers.1.weight', 'layers.1.bias']


### 5.3 buffer：不需要梯度但属于模块状态的张量

`self.register_buffer(name, tensor, persistent=True)`：

- 会随 `module.to(device)` 一起移动设备
- 不会出现在 `parameters()` 中（优化器不会更新它）
- `persistent=True` 时出现在 `state_dict()` 中；`persistent=False` 时不保存（适合可以重新计算出来的缓存，比如预先算好的三角函数表）

In [19]:
class WithCache(nn.Module):
    def __init__(self, n):
        super().__init__()
        self.register_buffer("table", torch.arange(n).float() ** 2, persistent=False)
        self.register_buffer("steps", torch.tensor(0))

    def forward(self, i):
        self.steps += 1
        return self.table[i]

m = WithCache(5)
print(m(torch.tensor([1, 3])), m.steps)
print("params:", list(m.parameters()))
print("state_dict keys:", list(m.state_dict().keys()))   # 没有 table

tensor([1., 9.]) tensor(1)
params: []
state_dict keys: ['steps']


### 5.4 `state_dict` / `load_state_dict`

`state_dict()` 是一个 `{名字: 张量}` 的有序字典，名字由属性路径用 `.` 连接而成。
**测试 adapter 会通过 `load_state_dict` 把参考权重灌进你的模块**，所以属性名、形状必须与要求一致。

In [20]:
class Tiny(nn.Module):
    def __init__(self):
        super().__init__()
        self.proj = nn.Linear(3, 2, bias=False)
        self.gain = nn.Parameter(torch.ones(2))

    def forward(self, x):
        return self.proj(x) * self.gain

a, b = Tiny(), Tiny()
print({k: tuple(v.shape) for k, v in a.state_dict().items()})
b.load_state_dict(a.state_dict())
x = torch.randn(4, 3)
torch.testing.assert_close(a(x), b(x))

# 名字不匹配时会报错——这能帮你尽早发现属性命名问题
try:
    b.load_state_dict({"proj.W": torch.zeros(2, 3), "gain": torch.ones(2)})
except RuntimeError as e:
    print(str(e).splitlines()[0])

{'gain': (2,), 'proj.weight': (2, 3)}
Error(s) in loading state_dict for Tiny:


**练习对应：** `exercises/modules.py` 中的 `Affine`、`TwoLayerMLP`、`ResidualMLPStack`、`RunningMean`。

## 6. autograd 与最小训练循环

- `requires_grad=True` 的张量参与的运算会被记录到计算图
- 对**标量** `loss` 调用 `loss.backward()`，梯度累加到各叶子张量的 `.grad`
- 梯度是**累加**的，所以每一步都要先清零（`optimizer.zero_grad()`）
- `with torch.no_grad():` 内的运算不记录计算图（推理、手动更新参数时使用）
- `x.detach()` 得到一个与计算图断开的新张量

In [21]:
x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
y = (x ** 3).sum()
y.backward()
print(x.grad)          # 3x^2 = [3, 12, 27]

y = (x ** 3).sum()
y.backward()
print(x.grad)          # 累加了！[6, 24, 54]
x.grad = None

tensor([ 3., 12., 27.])
tensor([ 6., 24., 54.])


### 6.1 手写 SGD vs 使用 `torch.optim`

下面用一个一维例子拟合 `y = 0.5 x - 1`。两种写法做的是同一件事。
作业后面会让你自己实现一个优化器（继承 `torch.optim.Optimizer`），理解手写版本很有帮助。

In [22]:
torch.manual_seed(0)
xs = torch.randn(100, 1)
ys = 0.5 * xs - 1

# --- 写法一：完全手写 ---
w = torch.zeros(1, requires_grad=True)
b = torch.zeros(1, requires_grad=True)
for step in range(200):
    pred = xs[:, 0] * w + b
    loss = ((pred - ys[:, 0]) ** 2).mean()
    loss.backward()
    with torch.no_grad():
        for p in (w, b):
            p -= 0.1 * p.grad
            p.grad = None
print("manual:", w.item(), b.item())

# --- 写法二：nn.Module + optimizer ---
model = Scaler(0.0)                                    # 只学一个缩放（演示用）
bias = nn.Parameter(torch.zeros(1))
opt = torch.optim.SGD(list(model.parameters()) + [bias], lr=0.1)
for step in range(200):
    loss = ((model(xs[:, 0]) + bias - ys[:, 0]) ** 2).mean()
    opt.zero_grad()
    loss.backward()
    opt.step()
print("optim :", model.alpha.item(), bias.item())

manual: 0.4999999403953552 -0.9999998807907104
optim : 0.4999999403953552 -0.9999998807907104


**练习对应：** `exercises/training.py`。

## 7. jaxtyping 标注与调试习惯

作业代码大量使用 `jaxtyping` 标注张量形状，例如 `Float[Tensor, "batch seq d_model"]`。
默认情况下它**只是文档**，不会在运行时检查，但能让你和读代码的人一眼看懂形状约定。

In [23]:
from jaxtyping import Float
from torch import Tensor

def scale_rows(x: Float[Tensor, "n d"], s: Float[Tensor, " n"]) -> Float[Tensor, "n d"]:
    out = x * s[:, None]
    assert out.shape == x.shape, f"{out.shape=} {x.shape=}"   # 便宜又有效的检查
    return out

print(scale_rows(torch.ones(3, 2), torch.tensor([1., 2., 3.])))

tensor([[1., 1.],
        [2., 2.],
        [3., 3.]])


### 调试清单（强烈建议养成习惯）

1. **用小形状、互不相同的维度**做实验，例如 `(2, 3, 5)` 而不是 `(4, 4, 4)`——维度搞反时立刻报错，而不是悄悄算错。
2. 在关键位置加 `assert x.shape == (...)`。
3. 用 `torch.arange` 构造输入，打印出来人工核对每个元素去了哪里。
4. 先写一个"笨"的 for 循环版本，再用 `torch.testing.assert_close` 和向量化版本对比（本章 tests 就是这么写的）。
5. 数值出现 `nan/inf` 时，逐层打印 `x.abs().max()` 定位。
6. 注意 dtype：整数张量做除法、`float16` 下的溢出等。

---

## 下一步

1. 完成 `warmup/exercises/` 下的三个文件
2. 运行 `uv run pytest warmup/tests -v` 直到全部通过
3. 回到 handout，开始正式作业。卡住时可以问 AI 助手——`AGENTS.md` 中写明了它可以如何帮助你